# Pipeline Completo — Carteira de Investimentos

Demonstra o pipeline end-to-end do carteira_auto:
**Carteira → Precos → Analise → Risco → Commodities → Rebalanceamento → Alertas**

Cobre:
1. Construir portfolio programaticamente
2. Executar pipeline completo via DAGEngine
3. Analisar resultados (metricas, risco, commodities)
4. Gerar recomendacoes de rebalanceamento
5. Avaliar alertas

**Nota**: Este notebook faz chamadas reais a APIs (Yahoo, BCB). Nao requer planilha Excel.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

## 1. Construir Portfolio

Em producao, o portfolio e carregado da planilha Excel via `LoadPortfolioNode`.
Aqui construimos programaticamente para demonstracao.

In [ ]:
from carteira_auto.core.models import Asset, Portfolio
from carteira_auto.core.engine import PipelineContext

# Portfolio diversificado de exemplo
portfolio = Portfolio(assets=[
    # Acoes
    Asset(ticker="PETR4", nome="Petrobras PN", classe="Acoes",
          posicao_atual=8000, preco_atual=35.50, preco_medio=28.00,
          pct_meta=0.15, n_cotas_atual=225, proventos_recebidos=450),
    Asset(ticker="VALE3", nome="Vale ON", classe="Acoes",
          posicao_atual=6000, preco_atual=62.00, preco_medio=55.00,
          pct_meta=0.12, n_cotas_atual=97, proventos_recebidos=300),
    Asset(ticker="ITUB4", nome="Itau PN", classe="Acoes",
          posicao_atual=5000, preco_atual=32.00, preco_medio=30.00,
          pct_meta=0.10, n_cotas_atual=156, proventos_recebidos=200),
    Asset(ticker="WEGE3", nome="WEG ON", classe="Acoes",
          posicao_atual=4000, preco_atual=45.00, preco_medio=40.00,
          pct_meta=0.08, n_cotas_atual=89, proventos_recebidos=100),
    # Renda Fixa
    Asset(ticker="TESOURO_SELIC", nome="Tesouro Selic 2029", classe="Renda Fixa",
          posicao_atual=15000, preco_atual=15000, preco_medio=14000,
          pct_meta=0.30, n_cotas_atual=1),
    # FIIs
    Asset(ticker="HGLG11", nome="CSHG Logistica", classe="FIIs",
          posicao_atual=3000, preco_atual=165.00, preco_medio=160.00,
          pct_meta=0.10, n_cotas_atual=18, proventos_recebidos=180),
    Asset(ticker="XPML11", nome="XP Malls", classe="FIIs",
          posicao_atual=2500, preco_atual=110.00, preco_medio=105.00,
          pct_meta=0.08, n_cotas_atual=23, proventos_recebidos=150),
])

total = sum(a.posicao_atual for a in portfolio.assets)
print(f"Portfolio: {len(portfolio.assets)} ativos, total R${total:,.0f}")
for a in portfolio.assets:
    pct = a.posicao_atual / total * 100
    print(f"  {a.ticker:15s} {a.classe:12s} R${a.posicao_atual:>10,.0f} ({pct:.1f}%)")

## 2. Executar Analyzers Independentes

Analyzers sem dependencias (deps=[]) podem ser executados diretamente.
Cada um busca dados das APIs e produz metricas no contexto.

In [ ]:
from carteira_auto.analyzers import MacroAnalyzer, CommodityAnalyzer, FiscalAnalyzer

ctx = PipelineContext()
ctx["portfolio"] = portfolio

# Macro (BCB + IBGE)
macro = MacroAnalyzer()
ctx = macro.run(ctx)
mc = ctx["macro_context"]
print(f"Macro: Selic={mc.selic}%, IPCA={mc.ipca}%, PTAX=R${mc.ptax_compra}")
print(f"  {mc.summary}")

In [ ]:
# Commodities (Yahoo)
commodity = CommodityAnalyzer()
ctx = commodity.run(ctx)
cm = ctx["commodity_metrics"]
print(f"Commodities: Brent=US${cm.oil_brent}, Ouro=US${cm.gold}")
print(f"  Ciclo: {cm.cycle_signal}")
print(f"  {cm.summary}")

In [ ]:
# Fiscal (BCB)
fiscal = FiscalAnalyzer()
ctx = fiscal.run(ctx)
fm = ctx["fiscal_metrics"]
print(f"Fiscal: divida={fm.divida_bruta_pib}% PIB, trajetoria={fm.fiscal_trajectory}")
print(f"  {fm.summary}")

In [ ]:
# Verificar erros parciais acumulados
if ctx.has_errors:
    print("\nErros parciais durante execucao:")
    for node, msg in ctx.errors.items():
        print(f"  {node}: {msg[:80]}..." if len(msg) > 80 else f"  {node}: {msg}")
else:
    print("\nNenhum erro parcial — todos os indicadores obtidos com sucesso")

## 3. Pipeline via Registry (CLI equivalente)

Em producao, pipelines sao executados via CLI:
```bash
python -m carteira_auto run macro
python -m carteira_auto run fiscal
python -m carteira_auto run commodities
```

Equivalente em Python:

In [ ]:
from carteira_auto.core.registry import create_engine, get_terminal_node, list_pipelines

# Listar todos os pipelines disponiveis
print(f"{len(list_pipelines())} pipelines disponiveis:")
for nome, descricao in sorted(list_pipelines().items()):
    print(f"  {nome:35s} | {descricao}")

In [ ]:
# Executar pipeline "fiscal" via engine
engine = create_engine()
terminal = get_terminal_node("fiscal")

# Dry-run primeiro
plan = engine.dry_run(terminal)
print(f"Pipeline 'fiscal' ({terminal}):")
print(f"  Plano: {' -> '.join(plan)}")

# Executar
ctx_fiscal = engine.run(terminal)
print(f"  Resultado: {ctx_fiscal['fiscal_metrics'].summary}")

## 4. MarketAnalyzer — Benchmarks

In [ ]:
from carteira_auto.analyzers import MarketAnalyzer

market = MarketAnalyzer()
ctx_market = PipelineContext()
ctx_market = market.run(ctx_market)

mm = ctx_market["market_metrics"]
print("=== Benchmarks de Mercado ===")
print(f"IBOV: {mm.ibov}")
print(f"IFIX: {mm.ifix}")
print(f"CDI acum. 12m: {mm.cdi_acumulado}")
print(f"S&P 500: {mm.sp500}")
print(f"\n{mm.summary}")

## 5. Visao Consolidada

Reunir todos os resultados em uma visao unica do cenario.

In [ ]:
print("=" * 60)
print("CENARIO CONSOLIDADO")
print("=" * 60)

print(f"\nMACRO BRASIL")
print(f"  Selic: {mc.selic}% | IPCA: {mc.ipca}%")
print(f"  PTAX: R${mc.ptax_compra} | PIB: {mc.pib_growth}")

print(f"\nFISCAL")
print(f"  Divida/PIB: {fm.divida_bruta_pib}% | Primario: {fm.resultado_primario_pib}%")
print(f"  Trajetoria: {fm.fiscal_trajectory}")

print(f"\nCOMMODITIES")
print(f"  Brent: US${cm.oil_brent} | Ouro: US${cm.gold}")
print(f"  Ciclo: {cm.cycle_signal}")

print(f"\nMERCADO")
print(f"  IBOV: {mm.ibov} | IFIX: {mm.ifix}")

print(f"\nPORTFOLIO")
print(f"  {len(portfolio.assets)} ativos, R${total:,.0f}")
print("=" * 60)

## Resumo

Este notebook demonstrou o pipeline completo do carteira_auto:

1. **Portfolio** construido com Pydantic models validados
2. **Analyzers** executados independentemente (macro, fiscal, commodities, mercado)
3. **Registry** para executar pipelines nomeados (equivalente ao CLI)
4. **Error handling** parcial — falhas individuais nao interrompem a analise
5. **Visao consolidada** reunindo todos os contextos

Proximos passos (fases futuras):
- **Fase 3**: Estrategias de alocacao + otimizacao (PyPortfolioOpt)
- **Fase 4**: ML scoring fundamentalista
- **Fase 5**: NLP / sentimento / geopolitica
- **Fase 6**: AI Reasoning (Claude integrado)
- **Fase 7**: Publishers (Telegram, PDF, email)